In [ ]:
%reset

In [ ]:
import sys
print(sys.executable)

In [ ]:
import scanpy as sc
import muon as mu
import pandas as pd
import numpy as np

In [ ]:
data_dir = "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/2.preprocessing_tonys_pipeline/2.normalized_data/normalized_data"
working_dir = "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks"


### Test

In [ ]:
d0_mdata = mu.read(f"{data_dir}/d0_normalized_gex_and_guide.h5mu")

In [ ]:
d0_mdata

In [ ]:
d0_mdata['GEX']

In [ ]:
d0_mdata['GEX'].layers["raw_counts"]

In [ ]:
d0_mdata['GEX'].var_names

In [ ]:
d0_mdata['guide'].var_names

In [ ]:
d0_mdata['guide']

In [ ]:
d0_mdata['guide'].obs["stype"]

In [ ]:
d0_mdata['GEX'].var['symbol']

In [ ]:
guide_meta_df = pd.read_csv(f"{working_dir}/300genes_guide_metadata_v43.tsv", sep='\t')

In [ ]:
guide_meta_df

In [ ]:
import scipy.io as sio
import pandas as pd
import os

### Day 0 

In [ ]:
mdata = d0_mdata

# Filter out cells that have no guides
#mdata = mdata[mdata['GEX'].obs['guides_per_cell'] > 0]

In [ ]:
# Define a temporary folder for transfer
temp_dir = working_dir + "/d0_sceptre_transfer"
os.makedirs(temp_dir, exist_ok=True)

# 1. EXTRACT MATRICES (Transpose them!)
# Scanpy is (Cells × Genes), Sceptre needs (Genes × Cells)
print("Writing sparse matrices...")
sio.mmwrite(f"{temp_dir}/response_matrix.mtx", mdata['GEX'].layers['raw_counts'].T)
sio.mmwrite(f"{temp_dir}/grna_matrix.mtx", mdata['guide'].X.T)

# 2. EXTRACT NAMES (Row/Col names are lost in .mtx)
# Save Gene names and Guide names to assign them later in R
pd.Series(mdata['GEX'].var_names).to_csv(f"{temp_dir}/gene_names.csv", index=False, header=False)
pd.Series(mdata['guide'].var_names).to_csv(f"{temp_dir}/guide_names.csv", index=False, header=False)
pd.Series(mdata['GEX'].obs_names).to_csv(f"{temp_dir}/cell_names.csv", index=False, header=False)

In [ ]:
mdata['GEX']

In [ ]:
# 3. EXTRACT COVARIATES 
rna = mdata['GEX']
covariates_df = pd.DataFrame({
    "batch": rna.obs['sample_id'],
    "grna_n_nonzero": rna.obs['guides_per_cell'],
    "grna_n_umis": rna.obs['guide_umi_counts'],
    "response_n_nonzero": rna.obs['n_genes_by_counts'],
    "response_n_umis": rna.obs['total_counts'],
    "response_p_mito": rna.obs['pct_counts_mt'],
})
covariates_df.to_csv(f"{temp_dir}/covariates.csv")
covariates_df

In [ ]:
# 4. PREPARE GUIDE MAPPING 
# Sceptre needs a table mapping 'grna_id' to 'grna_target'.

# 1. Create a "skeleton" DataFrame from the actual matrix order
# This ensures the order matches mdata['guide'].X perfectly
# We name the index 'grna_id' to match Sceptre's expectation immediately
guide_mapping = pd.DataFrame(index=mdata['guide'].var_names)
guide_mapping.index.name = 'grna_id'

# 2. Prepare metadata for merging
# Ensure 'guide_id' is the index so we can map efficiently
lookup_df = guide_meta_df.set_index('guide_id')

# 3. Map the targets onto the skeleton
# We use .loc[] to pull the specific target for each guide in the exact order required
guide_mapping['grna_target'] = lookup_df.loc[guide_mapping.index, 'intended_target_name']

# 4. Reset index so 'grna_id' becomes a column (Sceptre requires this)
guide_mapping = guide_mapping.reset_index()

# ==========================================
# VALIDATION CHECKS 
# ==========================================

# Check 1: Are there any NAs? (Guides in mdata that were missing from meta_df)
na_count = guide_mapping['grna_target'].isna().sum()
if na_count > 0:
    print(f"⚠️ WARNING: {na_count} guides have NaN targets! These will crash Sceptre.")
else:
    print("✅ All guides mapped successfully.")

# Check 2: Verify the first few rows manually
print("\nFirst 5 rows of mapping (Must match mdata['guide'].var_names order):")
print(guide_mapping.head())

# 5. Save for Sceptre
guide_mapping.to_csv(f"{temp_dir}/guide_mapping.csv", index=False)

In [ ]:
guide_mapping

In [ ]:
# Create a mapping from the Gene Symbol (which matches your guides)
# to the Gene Index (which matches the matrix rownames)

# Check which column in mdata['GEX'].var holds the symbol
symbol_col = 'symbol' 

gene_map = mdata['GEX'].var[[symbol_col]].reset_index()
gene_map.columns = ['response_id', 'grna_target'] # Rename for easy merging in R

# Save it
gene_map.to_csv(f"{temp_dir}/gene_symbol_map.csv", index=False)
print("✅ Gene mapping saved.")

In [ ]:
gene_map

#### Import data to SCEPTRE (Rcell)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(sceptre)
library(Matrix)

# Define path
data_dir <- "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/d0_sceptre_transfer"

# 1. LOAD MATRICES
cat("Loading matrices into R...\n")
response_mat <- readMM(file.path(data_dir, "response_matrix.mtx"))
grna_mat <- readMM(file.path(data_dir, "grna_matrix.mtx"))

# 2. ATTACH NAMES
gene_names <- read.csv(file.path(data_dir, "gene_names.csv"), header=FALSE)$V1
guide_names <- read.csv(file.path(data_dir, "guide_names.csv"), header=FALSE)$V1
cell_names <- read.csv(file.path(data_dir, "cell_names.csv"), header=FALSE)$V1

rownames(response_mat) <- gene_names
colnames(response_mat) <- cell_names

rownames(grna_mat) <- guide_names
colnames(grna_mat) <- cell_names

In [ ]:
%%R

# 3. LOAD METADATA
covariates <- read.csv(file.path(data_dir, "covariates.csv"), row.names=1)
guide_mapping <- read.csv(file.path(data_dir, "guide_mapping.csv"))

# ==============================================================================
# FIX: RENAME 'response_p_mito' SO WE KEEP IT
# Sceptre blocks the name 'response_p_mito' but doesn't calculate it.
# We rename it to 'mt_percent' so we can use it as a covariate later.
# ==============================================================================

# 1. Rename response_p_mito if it exists
if ("response_p_mito" %in% colnames(covariates)) {
    cat("Renaming 'response_p_mito' to 'mt_percent'...\n")
    names(covariates)[names(covariates) == "response_p_mito"] <- "mt_percent"
}

# 2. DROP OTHER RESERVED COLUMNS (which Sceptre DOES calculate)
# We still drop n_umis and n_nonzero because Sceptre re-calculates those automatically.
reserved_cols <- c("response_n_nonzero", "response_n_umis", "grna_n_nonzero", "grna_n_umis")
covariates <- covariates[, !(colnames(covariates) %in% reserved_cols), drop=FALSE]

# 4. CREATE SCEPTRE OBJECT
sceptre_object <- import_data(
  response_matrix = response_mat,
  grna_matrix = grna_mat,
  grna_target_data_frame = guide_mapping,
  extra_covariates = covariates,
  moi = "high"
)

print("✅ Sceptre object created!")
print(head(sceptre_object@covariate_data_frame))

In [ ]:
%%R
# 1. Load the translation map
gene_map <- read.csv(file.path(data_dir, "gene_symbol_map.csv"))

# 2. Start with guide mapping (which has 'grna_target' as Symbol)
discovery_pairs_highmoi <- guide_mapping

# 3. Merge to attach the correct response_id (Gene ID)
#    This matches 'grna_target' in guides to 'grna_target' in the gene map
discovery_pairs_highmoi <- merge(discovery_pairs_highmoi, gene_map, by="grna_target")

# 4. Select ONLY the two columns Sceptre needs
#    (We filter out non-targeting guides automatically via the merge)
discovery_pairs_highmoi <- discovery_pairs_highmoi[, c("grna_target", "response_id")]

# 5. DEDUPLICATE
#    Sceptre performs analysis at the *Target* level, not the *Guide* level.
#    We only need one row per Target-Gene pair.
discovery_pairs_highmoi <- unique(discovery_pairs_highmoi)

print(paste("Unique pairs to test:", nrow(discovery_pairs_highmoi)))
print(head(discovery_pairs_highmoi))

In [ ]:
%%R 

sceptre_object_highmoi <- set_analysis_parameters(
  sceptre_object = sceptre_object,
  discovery_pairs = discovery_pairs_highmoi,
  side = "both", control_group = "complement")

In [ ]:
%%R

png(file.path(data_dir, "grna_count_distributions.png"), width = 3000, height = 2000, res = 300)

print(plot_grna_count_distributions(
  sceptre_object = sceptre_object_highmoi,
  n_grnas_to_plot = 8
))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "grna_count_distributions.png"))

In [ ]:
%%R

plot_covariates(sceptre_object_highmoi, p_mito_threshold = 0.25)

In [ ]:
%%R
sceptre_object_highmoi <- assign_grnas(
  sceptre_object = sceptre_object_highmoi,
  method = "mixture",
  parallel = FALSE
)

In [ ]:
%%R
library(Matrix)

# 1. Get the assignment matrix
A <- get_grna_assignments(sceptre_object_highmoi)

# 2. Ensure it is a numeric sparse matrix (1/0)
if (inherits(A, "Matrix") || is.matrix(A)) {
    A_num <- if (inherits(A, "Matrix")) drop0(A * 1) else (A * 1)
} else {
    stop("Output is not a matrix.")
}

# --- FIX: ROBUST NAME RECOVERY ---
# We retrieve names using Sceptre's standard metadata accessors
# instead of touching internal slots like @grna_matrix.

# Try to get cell names from the matrix first
current_cell_names <- colnames(A_num)
current_guide_names <- rownames(A_num)

# If missing, grab them from the main object's covariate data frame
if (is.null(current_cell_names)) {
    message("⚠️ Matrix missing colnames. Fetching from sceptre_object...")
    # The covariate data frame ALWAYS has the cell IDs as rownames
    current_cell_names <- rownames(sceptre_object_highmoi@covariate_data_frame)
    
    # Safety check: lengths must match
    if (length(current_cell_names) != ncol(A_num)) {
        stop(paste("Mismatch! Matrix has", ncol(A_num), "cols but object has", length(current_cell_names), "cells."))
    }
    colnames(A_num) <- current_cell_names
}

# If guide names are missing, they should match the unique values in your guide mapping
if (is.null(current_guide_names)) {
    message("⚠️ Matrix missing rownames. Fetching from guide mapping...")
    # Get unique targets from the mapping table you loaded earlier
    # Note: This assumes the matrix rows are sorted/aligned to these unique targets.
    # In 'mixture' method, rows usually correspond to the 'grna_target' labels.
    
    # SAFE FALLBACK: If we can't be 100% sure of order, we warn the user.
    # But usually, they match the 'grna_target' column from the input.
    unique_targets <- sort(unique(guide_mapping$grna_target))
    
    if (length(unique_targets) == nrow(A_num)) {
         rownames(A_num) <- unique_targets
    } else {
         warning("Could not automatically align target names. Using generic IDs.")
         rownames(A_num) <- paste0("target_", seq_len(nrow(A_num)))
    }
}
# ---------------------------------

# 3. Convert to Long Format
s <- summary(A_num)
assign_long <- data.frame(
    cell_id = colnames(A_num)[s$j],
    grna_target = rownames(A_num)[s$i],
    value = s$x
)

# 4. Save
out_path <- file.path(data_dir, "grna_assignments_long.csv")
write.csv(assign_long, out_path, row.names = FALSE)

message("✅ Saved long CSV to: ", out_path)
print(head(assign_long))

In [ ]:
%%R
png(file.path(data_dir, "MOI.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "MOI.png"))

In [ ]:
%%R
sceptre_object_highmoi <- run_qc(
  sceptre_object = sceptre_object_highmoi,
  p_mito_threshold = 0.25
)

png(file.path(data_dir, "QC.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "QC.png"))

In [ ]:
%%R
# Calibration check


sceptre_object_highmoi <- run_calibration_check(
  sceptre_object = sceptre_object_highmoi,
  parallel = FALSE
)

png(file.path(data_dir, "calibration_check.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "calibration_check.png"))


In [ ]:
%%R
calibration_result <- get_result(
  sceptre_object = sceptre_object_highmoi,
  analysis = "run_calibration_check"
)
head(calibration_result)

In [ ]:
%%R
sceptre_object_highmoi <- run_discovery_analysis(
  sceptre_object = sceptre_object_highmoi,
  parallel = FALSE
)

png(file.path(data_dir, "discovery.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "discovery.png"))

In [ ]:
%%R
discovery_result <- get_result(
  sceptre_object = sceptre_object_highmoi,
  analysis = "run_discovery_analysis"
)

write.csv(discovery_result, file.path(data_dir, "TargetKD.csv"), row.names = FALSE)
message("✅ Saved discovery to: ", file.path(data_dir, "TargetKD.csv"))


### Day 1

In [ ]:
import scipy.io as sio
import pandas as pd
import os

In [ ]:
d1_mdata = mu.read(f"{data_dir}/d1_normalized_gex_and_guide.h5mu")
mdata = d1_mdata

# Filter out cells that have no guides
#mdata = mdata[mdata['GEX'].obs['guides_per_cell'] > 0]

In [ ]:
# Define a temporary folder for transfer
temp_dir = working_dir + "/d1_sceptre_transfer"
os.makedirs(temp_dir, exist_ok=True)

# 1. EXTRACT MATRICES (Transpose them!)
# Scanpy is (Cells × Genes), Sceptre needs (Genes × Cells)
print("Writing sparse matrices...")
sio.mmwrite(f"{temp_dir}/response_matrix.mtx", mdata['GEX'].layers['raw_counts'].T)
sio.mmwrite(f"{temp_dir}/grna_matrix.mtx", mdata['guide'].X.T)

# 2. EXTRACT NAMES (Row/Col names are lost in .mtx)
# Save Gene names and Guide names to assign them later in R
pd.Series(mdata['GEX'].var_names).to_csv(f"{temp_dir}/gene_names.csv", index=False, header=False)
pd.Series(mdata['guide'].var_names).to_csv(f"{temp_dir}/guide_names.csv", index=False, header=False)
pd.Series(mdata['GEX'].obs_names).to_csv(f"{temp_dir}/cell_names.csv", index=False, header=False)

In [ ]:
mdata['GEX']

In [ ]:
# 3. EXTRACT COVARIATES 
rna = mdata['GEX']
covariates_df = pd.DataFrame({
    "batch": rna.obs['sample_id'],
    "grna_n_nonzero": rna.obs['guides_per_cell'],
    "grna_n_umis": rna.obs['guide_umi_counts'],
    "response_n_nonzero": rna.obs['n_genes_by_counts'],
    "response_n_umis": rna.obs['total_counts'],
    "response_p_mito": rna.obs['pct_counts_mt'],
})
covariates_df.to_csv(f"{temp_dir}/covariates.csv")
covariates_df

In [ ]:
guide_meta_df = pd.read_csv(f"{working_dir}/300genes_guide_metadata_v43.tsv", sep='\t')

In [ ]:
# 4. PREPARE GUIDE MAPPING 
# Sceptre needs a table mapping 'grna_id' to 'grna_target'.

# 1. Create a "skeleton" DataFrame from the actual matrix order
# This ensures the order matches mdata['guide'].X perfectly
# We name the index 'grna_id' to match Sceptre's expectation immediately
guide_mapping = pd.DataFrame(index=mdata['guide'].var_names)
guide_mapping.index.name = 'grna_id'

# 2. Prepare metadata for merging
# Ensure 'guide_id' is the index so we can map efficiently
lookup_df = guide_meta_df.set_index('guide_id')

# 3. Map the targets onto the skeleton
# We use .loc[] to pull the specific target for each guide in the exact order required
guide_mapping['grna_target'] = lookup_df.loc[guide_mapping.index, 'intended_target_name']

# 4. Reset index so 'grna_id' becomes a column (Sceptre requires this)
guide_mapping = guide_mapping.reset_index()

# ==========================================
# VALIDATION CHECKS 
# ==========================================

# Check 1: Are there any NAs? (Guides in mdata that were missing from meta_df)
na_count = guide_mapping['grna_target'].isna().sum()
if na_count > 0:
    print(f"⚠️ WARNING: {na_count} guides have NaN targets! These will crash Sceptre.")
else:
    print("✅ All guides mapped successfully.")

# Check 2: Verify the first few rows manually
print("\nFirst 5 rows of mapping (Must match mdata['guide'].var_names order):")
print(guide_mapping.head())

# 5. Save for Sceptre
guide_mapping.to_csv(f"{temp_dir}/guide_mapping.csv", index=False)

In [ ]:
guide_mapping

In [ ]:
# Create a mapping from the Gene Symbol (which matches your guides)
# to the Gene Index (which matches the matrix rownames)

# Check which column in mdata['GEX'].var holds the symbol
symbol_col = 'symbol' 

gene_map = mdata['GEX'].var[[symbol_col]].reset_index()
gene_map.columns = ['response_id', 'grna_target'] # Rename for easy merging in R

# Save it
gene_map.to_csv(f"{temp_dir}/gene_symbol_map.csv", index=False)
print("✅ Gene mapping saved.")

In [ ]:
gene_map

#### Import data to SCEPTRE (Rcell)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(sceptre)
library(Matrix)

# Define path
data_dir <- "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/d1_sceptre_transfer"

# 1. LOAD MATRICES
cat("Loading matrices into R...\n")
response_mat <- readMM(file.path(data_dir, "response_matrix.mtx"))
grna_mat <- readMM(file.path(data_dir, "grna_matrix.mtx"))

# 2. ATTACH NAMES
gene_names <- read.csv(file.path(data_dir, "gene_names.csv"), header=FALSE)$V1
guide_names <- read.csv(file.path(data_dir, "guide_names.csv"), header=FALSE)$V1
cell_names <- read.csv(file.path(data_dir, "cell_names.csv"), header=FALSE)$V1

rownames(response_mat) <- gene_names
colnames(response_mat) <- cell_names

rownames(grna_mat) <- guide_names
colnames(grna_mat) <- cell_names

In [ ]:
%%R

# 3. LOAD METADATA
covariates <- read.csv(file.path(data_dir, "covariates.csv"), row.names=1)
guide_mapping <- read.csv(file.path(data_dir, "guide_mapping.csv"))

# ==============================================================================
# FIX: RENAME 'response_p_mito' SO WE KEEP IT
# Sceptre blocks the name 'response_p_mito' but doesn't calculate it.
# We rename it to 'mt_percent' so we can use it as a covariate later.
# ==============================================================================

# 1. Rename response_p_mito if it exists
if ("response_p_mito" %in% colnames(covariates)) {
    cat("Renaming 'response_p_mito' to 'mt_percent'...\n")
    names(covariates)[names(covariates) == "response_p_mito"] <- "mt_percent"
}

# 2. DROP OTHER RESERVED COLUMNS (which Sceptre DOES calculate)
# We still drop n_umis and n_nonzero because Sceptre re-calculates those automatically.
reserved_cols <- c("response_n_nonzero", "response_n_umis", "grna_n_nonzero", "grna_n_umis")
covariates <- covariates[, !(colnames(covariates) %in% reserved_cols), drop=FALSE]

# 4. CREATE SCEPTRE OBJECT
sceptre_object <- import_data(
  response_matrix = response_mat,
  grna_matrix = grna_mat,
  grna_target_data_frame = guide_mapping,
  extra_covariates = covariates,
  moi = "high"
)

print("✅ Sceptre object created!")
print(head(sceptre_object@covariate_data_frame))

In [ ]:
%%R
# 1. Load the translation map
gene_map <- read.csv(file.path(data_dir, "gene_symbol_map.csv"))

# 2. Start with guide mapping (which has 'grna_target' as Symbol)
discovery_pairs_highmoi <- guide_mapping

# 3. Merge to attach the correct response_id (Gene ID)
#    This matches 'grna_target' in guides to 'grna_target' in the gene map
discovery_pairs_highmoi <- merge(discovery_pairs_highmoi, gene_map, by="grna_target")

# 4. Select ONLY the two columns Sceptre needs
#    (We filter out non-targeting guides automatically via the merge)
discovery_pairs_highmoi <- discovery_pairs_highmoi[, c("grna_target", "response_id")]

# 5. DEDUPLICATE
#    Sceptre performs analysis at the *Target* level, not the *Guide* level.
#    We only need one row per Target-Gene pair.
discovery_pairs_highmoi <- unique(discovery_pairs_highmoi)

print(paste("Unique pairs to test:", nrow(discovery_pairs_highmoi)))
print(head(discovery_pairs_highmoi))

In [ ]:
%%R 

sceptre_object_highmoi <- set_analysis_parameters(
  sceptre_object = sceptre_object,
  discovery_pairs = discovery_pairs_highmoi,
  side = "both", control_group = "complement")

In [ ]:
%%R

png(file.path(data_dir, "grna_count_distributions.png"), width = 3000, height = 2000, res = 300)

print(plot_grna_count_distributions(
  sceptre_object = sceptre_object_highmoi,
  n_grnas_to_plot = 8
))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "grna_count_distributions.png"))

In [ ]:
%%R

plot_covariates(sceptre_object_highmoi, p_mito_threshold = 0.25)

In [ ]:
%%R
sceptre_object_highmoi <- assign_grnas(
  sceptre_object = sceptre_object_highmoi,
  method = "mixture",
  parallel = FALSE
)

In [ ]:
%%R
library(Matrix)

# 1. Get the assignment matrix
A <- get_grna_assignments(sceptre_object_highmoi)

# 2. Ensure it is a numeric sparse matrix (1/0)
if (inherits(A, "Matrix") || is.matrix(A)) {
    A_num <- if (inherits(A, "Matrix")) drop0(A * 1) else (A * 1)
} else {
    stop("Output is not a matrix.")
}

# --- FIX: ROBUST NAME RECOVERY ---
# We retrieve names using Sceptre's standard metadata accessors
# instead of touching internal slots like @grna_matrix.

# Try to get cell names from the matrix first
current_cell_names <- colnames(A_num)
current_guide_names <- rownames(A_num)

# If missing, grab them from the main object's covariate data frame
if (is.null(current_cell_names)) {
    message("⚠️ Matrix missing colnames. Fetching from sceptre_object...")
    # The covariate data frame ALWAYS has the cell IDs as rownames
    current_cell_names <- rownames(sceptre_object_highmoi@covariate_data_frame)
    
    # Safety check: lengths must match
    if (length(current_cell_names) != ncol(A_num)) {
        stop(paste("Mismatch! Matrix has", ncol(A_num), "cols but object has", length(current_cell_names), "cells."))
    }
    colnames(A_num) <- current_cell_names
}

# If guide names are missing, they should match the unique values in your guide mapping
if (is.null(current_guide_names)) {
    message("⚠️ Matrix missing rownames. Fetching from guide mapping...")
    # Get unique targets from the mapping table you loaded earlier
    # Note: This assumes the matrix rows are sorted/aligned to these unique targets.
    # In 'mixture' method, rows usually correspond to the 'grna_target' labels.
    
    # SAFE FALLBACK: If we can't be 100% sure of order, we warn the user.
    # But usually, they match the 'grna_target' column from your input.
    unique_targets <- sort(unique(guide_mapping$grna_target))
    
    if (length(unique_targets) == nrow(A_num)) {
         rownames(A_num) <- unique_targets
    } else {
         warning("Could not automatically align target names. Using generic IDs.")
         rownames(A_num) <- paste0("target_", seq_len(nrow(A_num)))
    }
}
# ---------------------------------

# 3. Convert to Long Format
s <- summary(A_num)
assign_long <- data.frame(
    cell_id = colnames(A_num)[s$j],
    grna_target = rownames(A_num)[s$i],
    value = s$x
)

# 4. Save
out_path <- file.path(data_dir, "grna_assignments_long.csv")
write.csv(assign_long, out_path, row.names = FALSE)

message("✅ Saved long CSV to: ", out_path)
print(head(assign_long))

In [ ]:
%%R
png(file.path(data_dir, "MOI.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "MOI.png"))

In [ ]:
%%R
sceptre_object_highmoi <- run_qc(
  sceptre_object = sceptre_object_highmoi,
  p_mito_threshold = 0.25
)

png(file.path(data_dir, "QC.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "QC.png"))

In [ ]:
%%R
# Calibration check


sceptre_object_highmoi <- run_calibration_check(
  sceptre_object = sceptre_object_highmoi,
  parallel = FALSE
)

png(file.path(data_dir, "calibration_check.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "calibration_check.png"))


In [ ]:
%%R
calibration_result <- get_result(
  sceptre_object = sceptre_object_highmoi,
  analysis = "run_calibration_check"
)
head(calibration_result)

In [ ]:
%%R
sceptre_object_highmoi <- run_discovery_analysis(
  sceptre_object = sceptre_object_highmoi,
  parallel = FALSE
)

png(file.path(data_dir, "discovery.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "discovery.png"))

In [ ]:
%%R
discovery_result <- get_result(
  sceptre_object = sceptre_object_highmoi,
  analysis = "run_discovery_analysis"
)

write.csv(discovery_result, file.path(data_dir, "TargetKD.csv"), row.names = FALSE)
message("✅ Saved discovery to: ", file.path(data_dir, "TargetKD.csv"))


### Day 2

In [ ]:
d2_mdata = mu.read(f"{data_dir}/d2_normalized_gex_and_guide.h5mu")
mdata = d2_mdata

# Filter out cells that have no guides
#mdata = mdata[mdata['GEX'].obs['guides_per_cell'] > 0]

In [ ]:
# Define a temporary folder for transfer
temp_dir = working_dir + "/d2_sceptre_transfer"
os.makedirs(temp_dir, exist_ok=True)

# 1. EXTRACT MATRICES (Transpose them!)
# Scanpy is (Cells × Genes), Sceptre needs (Genes × Cells)
print("Writing sparse matrices...")
sio.mmwrite(f"{temp_dir}/response_matrix.mtx", mdata['GEX'].layers['raw_counts'].T)
sio.mmwrite(f"{temp_dir}/grna_matrix.mtx", mdata['guide'].X.T)

# 2. EXTRACT NAMES (Row/Col names are lost in .mtx)
# Save Gene names and Guide names to assign them later in R
pd.Series(mdata['GEX'].var_names).to_csv(f"{temp_dir}/gene_names.csv", index=False, header=False)
pd.Series(mdata['guide'].var_names).to_csv(f"{temp_dir}/guide_names.csv", index=False, header=False)
pd.Series(mdata['GEX'].obs_names).to_csv(f"{temp_dir}/cell_names.csv", index=False, header=False)

In [ ]:
mdata['GEX']

In [ ]:
# 3. EXTRACT COVARIATES 
rna = mdata['GEX']
covariates_df = pd.DataFrame({
    "batch": rna.obs['sample_id'],
    "grna_n_nonzero": rna.obs['guides_per_cell'],
    "grna_n_umis": rna.obs['guide_umi_counts'],
    "response_n_nonzero": rna.obs['n_genes_by_counts'],
    "response_n_umis": rna.obs['total_counts'],
    "response_p_mito": rna.obs['pct_counts_mt'],
})
covariates_df.to_csv(f"{temp_dir}/covariates.csv")
covariates_df

In [ ]:
guide_meta_df = pd.read_csv(f"{working_dir}/300genes_guide_metadata_v43.tsv", sep='\t')

In [ ]:
# 4. PREPARE GUIDE MAPPING 
# Sceptre needs a table mapping 'grna_id' to 'grna_target'.

# 1. Create a "skeleton" DataFrame from the actual matrix order
# This ensures the order matches mdata['guide'].X perfectly
# We name the index 'grna_id' to match Sceptre's expectation immediately
guide_mapping = pd.DataFrame(index=mdata['guide'].var_names)
guide_mapping.index.name = 'grna_id'

# 2. Prepare metadata for merging
# Ensure 'guide_id' is the index so we can map efficiently
lookup_df = guide_meta_df.set_index('guide_id')

# 3. Map the targets onto the skeleton
# We use .loc[] to pull the specific target for each guide in the exact order required
guide_mapping['grna_target'] = lookup_df.loc[guide_mapping.index, 'intended_target_name']

# 4. Reset index so 'grna_id' becomes a column (Sceptre requires this)
guide_mapping = guide_mapping.reset_index()

# ==========================================
# VALIDATION CHECKS 
# ==========================================

# Check 1: Are there any NAs? (Guides in mdata that were missing from meta_df)
na_count = guide_mapping['grna_target'].isna().sum()
if na_count > 0:
    print(f"⚠️ WARNING: {na_count} guides have NaN targets! These will crash Sceptre.")
else:
    print("✅ All guides mapped successfully.")

# Check 2: Verify the first few rows manually
print("\nFirst 5 rows of mapping (Must match mdata['guide'].var_names order):")
print(guide_mapping.head())

# 5. Save for Sceptre
guide_mapping.to_csv(f"{temp_dir}/guide_mapping.csv", index=False)

In [ ]:
guide_mapping

In [ ]:
# Create a mapping from the Gene Symbol (which matches your guides)
# to the Gene Index (which matches the matrix rownames)

# Check which column in mdata['GEX'].var holds the symbol
symbol_col = 'symbol' 

gene_map = mdata['GEX'].var[[symbol_col]].reset_index()
gene_map.columns = ['response_id', 'grna_target'] # Rename for easy merging in R

# Save it
gene_map.to_csv(f"{temp_dir}/gene_symbol_map.csv", index=False)
print("✅ Gene mapping saved.")

In [ ]:
gene_map

#### Import data to SCEPTRE (Rcell)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(sceptre)
library(Matrix)

# Define path
data_dir <- "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/d2_sceptre_transfer"

# 1. LOAD MATRICES
cat("Loading matrices into R...\n")
response_mat <- readMM(file.path(data_dir, "response_matrix.mtx"))
grna_mat <- readMM(file.path(data_dir, "grna_matrix.mtx"))

# 2. ATTACH NAMES
gene_names <- read.csv(file.path(data_dir, "gene_names.csv"), header=FALSE)$V1
guide_names <- read.csv(file.path(data_dir, "guide_names.csv"), header=FALSE)$V1
cell_names <- read.csv(file.path(data_dir, "cell_names.csv"), header=FALSE)$V1

rownames(response_mat) <- gene_names
colnames(response_mat) <- cell_names

rownames(grna_mat) <- guide_names
colnames(grna_mat) <- cell_names

In [ ]:
%%R

# 3. LOAD METADATA
covariates <- read.csv(file.path(data_dir, "covariates.csv"), row.names=1)
guide_mapping <- read.csv(file.path(data_dir, "guide_mapping.csv"))

# ==============================================================================
# FIX: RENAME 'response_p_mito' SO WE KEEP IT
# Sceptre blocks the name 'response_p_mito' but doesn't calculate it.
# We rename it to 'mt_percent' so we can use it as a covariate later.
# ==============================================================================

# 1. Rename response_p_mito if it exists
if ("response_p_mito" %in% colnames(covariates)) {
    cat("Renaming 'response_p_mito' to 'mt_percent'...\n")
    names(covariates)[names(covariates) == "response_p_mito"] <- "mt_percent"
}

# 2. DROP OTHER RESERVED COLUMNS (which Sceptre DOES calculate)
# We still drop n_umis and n_nonzero because Sceptre re-calculates those automatically.
reserved_cols <- c("response_n_nonzero", "response_n_umis", "grna_n_nonzero", "grna_n_umis")
covariates <- covariates[, !(colnames(covariates) %in% reserved_cols), drop=FALSE]

# 4. CREATE SCEPTRE OBJECT
sceptre_object <- import_data(
  response_matrix = response_mat,
  grna_matrix = grna_mat,
  grna_target_data_frame = guide_mapping,
  extra_covariates = covariates,
  moi = "high"
)

print("✅ Sceptre object created!")
print(head(sceptre_object@covariate_data_frame))

In [ ]:
%%R
# 1. Load the translation map
gene_map <- read.csv(file.path(data_dir, "gene_symbol_map.csv"))

# 2. Start with guide mapping (which has 'grna_target' as Symbol)
discovery_pairs_highmoi <- guide_mapping

# 3. Merge to attach the correct response_id (Gene ID)
#    This matches 'grna_target' in guides to 'grna_target' in the gene map
discovery_pairs_highmoi <- merge(discovery_pairs_highmoi, gene_map, by="grna_target")

# 4. Select ONLY the two columns Sceptre needs
#    (We filter out non-targeting guides automatically via the merge)
discovery_pairs_highmoi <- discovery_pairs_highmoi[, c("grna_target", "response_id")]

# 5. DEDUPLICATE
#    Sceptre performs analysis at the *Target* level, not the *Guide* level.
#    We only need one row per Target-Gene pair.
discovery_pairs_highmoi <- unique(discovery_pairs_highmoi)

print(paste("Unique pairs to test:", nrow(discovery_pairs_highmoi)))
print(head(discovery_pairs_highmoi))

In [ ]:
%%R 

sceptre_object_highmoi <- set_analysis_parameters(
  sceptre_object = sceptre_object,
  discovery_pairs = discovery_pairs_highmoi,
  side = "both", control_group = "complement")

In [ ]:
%%R

png(file.path(data_dir, "grna_count_distributions.png"), width = 3000, height = 2000, res = 300)

print(plot_grna_count_distributions(
  sceptre_object = sceptre_object_highmoi,
  n_grnas_to_plot = 8
))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "grna_count_distributions.png"))

In [ ]:
%%R

plot_covariates(sceptre_object_highmoi, p_mito_threshold = 0.25)

In [ ]:
%%R
sceptre_object_highmoi <- assign_grnas(
  sceptre_object = sceptre_object_highmoi,
  method = "mixture",
  parallel = FALSE
)

In [ ]:
%%R
library(Matrix)

# 1. Get the assignment matrix
A <- get_grna_assignments(sceptre_object_highmoi)

# 2. Ensure it is a numeric sparse matrix (1/0)
if (inherits(A, "Matrix") || is.matrix(A)) {
    A_num <- if (inherits(A, "Matrix")) drop0(A * 1) else (A * 1)
} else {
    stop("Output is not a matrix.")
}

# --- FIX: ROBUST NAME RECOVERY ---
# We retrieve names using Sceptre's standard metadata accessors
# instead of touching internal slots like @grna_matrix.

# Try to get cell names from the matrix first
current_cell_names <- colnames(A_num)
current_guide_names <- rownames(A_num)

# If missing, grab them from the main object's covariate data frame
if (is.null(current_cell_names)) {
    message("⚠️ Matrix missing colnames. Fetching from sceptre_object...")
    # The covariate data frame ALWAYS has the cell IDs as rownames
    current_cell_names <- rownames(sceptre_object_highmoi@covariate_data_frame)
    
    # Safety check: lengths must match
    if (length(current_cell_names) != ncol(A_num)) {
        stop(paste("Mismatch! Matrix has", ncol(A_num), "cols but object has", length(current_cell_names), "cells."))
    }
    colnames(A_num) <- current_cell_names
}

# If guide names are missing, they should match the unique values in your guide mapping
if (is.null(current_guide_names)) {
    message("⚠️ Matrix missing rownames. Fetching from guide mapping...")
    # Get unique targets from the mapping table you loaded earlier
    # Note: This assumes the matrix rows are sorted/aligned to these unique targets.
    # In 'mixture' method, rows usually correspond to the 'grna_target' labels.
    
    # SAFE FALLBACK: If we can't be 100% sure of order, we warn the user.
    # But usually, they match the 'grna_target' column from your input.
    unique_targets <- sort(unique(guide_mapping$grna_target))
    
    if (length(unique_targets) == nrow(A_num)) {
         rownames(A_num) <- unique_targets
    } else {
         warning("Could not automatically align target names. Using generic IDs.")
         rownames(A_num) <- paste0("target_", seq_len(nrow(A_num)))
    }
}
# ---------------------------------

# 3. Convert to Long Format
s <- summary(A_num)
assign_long <- data.frame(
    cell_id = colnames(A_num)[s$j],
    grna_target = rownames(A_num)[s$i],
    value = s$x
)

# 4. Save
out_path <- file.path(data_dir, "grna_assignments_long.csv")
write.csv(assign_long, out_path, row.names = FALSE)

message("✅ Saved long CSV to: ", out_path)
print(head(assign_long))

In [ ]:
%%R
png(file.path(data_dir, "MOI.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "MOI.png"))

In [ ]:
%%R
sceptre_object_highmoi <- run_qc(
  sceptre_object = sceptre_object_highmoi,
  p_mito_threshold = 0.25
)

png(file.path(data_dir, "QC.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "QC.png"))

In [ ]:
%%R
# Calibration check


sceptre_object_highmoi <- run_calibration_check(
  sceptre_object = sceptre_object_highmoi,
  parallel = FALSE
)

png(file.path(data_dir, "calibration_check.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "calibration_check.png"))


In [ ]:
%%R
calibration_result <- get_result(
  sceptre_object = sceptre_object_highmoi,
  analysis = "run_calibration_check"
)
head(calibration_result)

In [ ]:
%%R
sceptre_object_highmoi <- run_discovery_analysis(
  sceptre_object = sceptre_object_highmoi,
  parallel = FALSE
)

png(file.path(data_dir, "discovery.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "discovery.png"))

In [ ]:
%%R
discovery_result <- get_result(
  sceptre_object = sceptre_object_highmoi,
  analysis = "run_discovery_analysis"
)

write.csv(discovery_result, file.path(data_dir, "TargetKD.csv"), row.names = FALSE)
message("✅ Saved discovery to: ", file.path(data_dir, "TargetKD.csv"))


### Day 3

In [ ]:
import scipy.io as sio
import pandas as pd
import os

In [ ]:
d3_mdata = mu.read(f"{data_dir}/d3_normalized_gex_and_guide.h5mu")
mdata = d3_mdata

# Filter out cells that have no guides
#mdata = mdata[mdata['GEX'].obs['guides_per_cell'] > 0]

In [ ]:
# Define a temporary folder for transfer
temp_dir = working_dir + "/d3_sceptre_transfer"
os.makedirs(temp_dir, exist_ok=True)

# 1. EXTRACT MATRICES (Transpose them!)
# Scanpy is (Cells × Genes), Sceptre needs (Genes × Cells)
print("Writing sparse matrices...")
sio.mmwrite(f"{temp_dir}/response_matrix.mtx", mdata['GEX'].layers['raw_counts'].T)
sio.mmwrite(f"{temp_dir}/grna_matrix.mtx", mdata['guide'].X.T)

# 2. EXTRACT NAMES (Row/Col names are lost in .mtx)
# Save Gene names and Guide names to assign them later in R
pd.Series(mdata['GEX'].var_names).to_csv(f"{temp_dir}/gene_names.csv", index=False, header=False)
pd.Series(mdata['guide'].var_names).to_csv(f"{temp_dir}/guide_names.csv", index=False, header=False)
pd.Series(mdata['GEX'].obs_names).to_csv(f"{temp_dir}/cell_names.csv", index=False, header=False)

In [ ]:
mdata['GEX']

In [ ]:
# 3. EXTRACT COVARIATES 
rna = mdata['GEX']
covariates_df = pd.DataFrame({
    "batch": rna.obs['sample_id'],
    "grna_n_nonzero": rna.obs['guides_per_cell'],
    "grna_n_umis": rna.obs['guide_umi_counts'],
    "response_n_nonzero": rna.obs['n_genes_by_counts'],
    "response_n_umis": rna.obs['total_counts'],
    "response_p_mito": rna.obs['pct_counts_mt'],
})
covariates_df.to_csv(f"{temp_dir}/covariates.csv")
covariates_df

In [ ]:
guide_meta_df = pd.read_csv(f"{working_dir}/300genes_guide_metadata_v43.tsv", sep='\t')

In [ ]:
# 4. PREPARE GUIDE MAPPING 
# Sceptre needs a table mapping 'grna_id' to 'grna_target'.

# 1. Create a "skeleton" DataFrame from the actual matrix order
# This ensures the order matches mdata['guide'].X perfectly
# We name the index 'grna_id' to match Sceptre's expectation immediately
guide_mapping = pd.DataFrame(index=mdata['guide'].var_names)
guide_mapping.index.name = 'grna_id'

# 2. Prepare metadata for merging
# Ensure 'guide_id' is the index so we can map efficiently
lookup_df = guide_meta_df.set_index('guide_id')

# 3. Map the targets onto the skeleton
# We use .loc[] to pull the specific target for each guide in the exact order required
guide_mapping['grna_target'] = lookup_df.loc[guide_mapping.index, 'intended_target_name']

# 4. Reset index so 'grna_id' becomes a column (Sceptre requires this)
guide_mapping = guide_mapping.reset_index()

# ==========================================
# VALIDATION CHECKS 
# ==========================================

# Check 1: Are there any NAs? (Guides in mdata that were missing from meta_df)
na_count = guide_mapping['grna_target'].isna().sum()
if na_count > 0:
    print(f"⚠️ WARNING: {na_count} guides have NaN targets! These will crash Sceptre.")
else:
    print("✅ All guides mapped successfully.")

# Check 2: Verify the first few rows manually
print("\nFirst 5 rows of mapping (Must match mdata['guide'].var_names order):")
print(guide_mapping.head())

# 5. Save for Sceptre
guide_mapping.to_csv(f"{temp_dir}/guide_mapping.csv", index=False)

In [ ]:
guide_mapping

In [ ]:
# Create a mapping from the Gene Symbol (which matches your guides)
# to the Gene Index (which matches the matrix rownames)

# Check which column in mdata['GEX'].var holds the symbol
symbol_col = 'symbol' 

gene_map = mdata['GEX'].var[[symbol_col]].reset_index()
gene_map.columns = ['response_id', 'grna_target'] # Rename for easy merging in R

# Save it
gene_map.to_csv(f"{temp_dir}/gene_symbol_map.csv", index=False)
print("✅ Gene mapping saved.")

In [ ]:
gene_map

#### Import data to SCEPTRE (Rcell)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(sceptre)
library(Matrix)

# Define path
data_dir <- "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/d3_sceptre_transfer"

# 1. LOAD MATRICES
cat("Loading matrices into R...\n")
response_mat <- readMM(file.path(data_dir, "response_matrix.mtx"))
grna_mat <- readMM(file.path(data_dir, "grna_matrix.mtx"))

# 2. ATTACH NAMES
gene_names <- read.csv(file.path(data_dir, "gene_names.csv"), header=FALSE)$V1
guide_names <- read.csv(file.path(data_dir, "guide_names.csv"), header=FALSE)$V1
cell_names <- read.csv(file.path(data_dir, "cell_names.csv"), header=FALSE)$V1

rownames(response_mat) <- gene_names
colnames(response_mat) <- cell_names

rownames(grna_mat) <- guide_names
colnames(grna_mat) <- cell_names

In [ ]:
%%R

# 3. LOAD METADATA
covariates <- read.csv(file.path(data_dir, "covariates.csv"), row.names=1)
guide_mapping <- read.csv(file.path(data_dir, "guide_mapping.csv"))

# ==============================================================================
# FIX: RENAME 'response_p_mito' SO WE KEEP IT
# Sceptre blocks the name 'response_p_mito' but doesn't calculate it.
# We rename it to 'mt_percent' so we can use it as a covariate later.
# ==============================================================================

# 1. Rename response_p_mito if it exists
if ("response_p_mito" %in% colnames(covariates)) {
    cat("Renaming 'response_p_mito' to 'mt_percent'...\n")
    names(covariates)[names(covariates) == "response_p_mito"] <- "mt_percent"
}

# 2. DROP OTHER RESERVED COLUMNS (which Sceptre DOES calculate)
# We still drop n_umis and n_nonzero because Sceptre re-calculates those automatically.
reserved_cols <- c("response_n_nonzero", "response_n_umis", "grna_n_nonzero", "grna_n_umis")
covariates <- covariates[, !(colnames(covariates) %in% reserved_cols), drop=FALSE]

# 4. CREATE SCEPTRE OBJECT
sceptre_object <- import_data(
  response_matrix = response_mat,
  grna_matrix = grna_mat,
  grna_target_data_frame = guide_mapping,
  extra_covariates = covariates,
  moi = "high"
)

print("✅ Sceptre object created!")
print(head(sceptre_object@covariate_data_frame))

In [ ]:
%%R
# 1. Load the translation map
gene_map <- read.csv(file.path(data_dir, "gene_symbol_map.csv"))

# 2. Start with guide mapping (which has 'grna_target' as Symbol)
discovery_pairs_highmoi <- guide_mapping

# 3. Merge to attach the correct response_id (Gene ID)
#    This matches 'grna_target' in guides to 'grna_target' in the gene map
discovery_pairs_highmoi <- merge(discovery_pairs_highmoi, gene_map, by="grna_target")

# 4. Select ONLY the two columns Sceptre needs
#    (We filter out non-targeting guides automatically via the merge)
discovery_pairs_highmoi <- discovery_pairs_highmoi[, c("grna_target", "response_id")]

# 5. DEDUPLICATE
#    Sceptre performs analysis at the *Target* level, not the *Guide* level.
#    We only need one row per Target-Gene pair.
discovery_pairs_highmoi <- unique(discovery_pairs_highmoi)

print(paste("Unique pairs to test:", nrow(discovery_pairs_highmoi)))
print(head(discovery_pairs_highmoi))

In [ ]:
%%R 

sceptre_object_highmoi <- set_analysis_parameters(
  sceptre_object = sceptre_object,
  discovery_pairs = discovery_pairs_highmoi,
  side = "both", control_group = "complement")

In [ ]:
%%R

png(file.path(data_dir, "grna_count_distributions.png"), width = 3000, height = 2000, res = 300)

print(plot_grna_count_distributions(
  sceptre_object = sceptre_object_highmoi,
  n_grnas_to_plot = 8
))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "grna_count_distributions.png"))

In [ ]:
%%R

plot_covariates(sceptre_object_highmoi, p_mito_threshold = 0.25)

In [ ]:
%%R
sceptre_object_highmoi <- assign_grnas(
  sceptre_object = sceptre_object_highmoi,
  method = "mixture",
  parallel = FALSE
)

In [ ]:
%%R
library(Matrix)

# 1. Get the assignment matrix
A <- get_grna_assignments(sceptre_object_highmoi)

# 2. Ensure it is a numeric sparse matrix (1/0)
if (inherits(A, "Matrix") || is.matrix(A)) {
    A_num <- if (inherits(A, "Matrix")) drop0(A * 1) else (A * 1)
} else {
    stop("Output is not a matrix.")
}

# --- FIX: ROBUST NAME RECOVERY ---
# We retrieve names using Sceptre's standard metadata accessors
# instead of touching internal slots like @grna_matrix.

# Try to get cell names from the matrix first
current_cell_names <- colnames(A_num)
current_guide_names <- rownames(A_num)

# If missing, grab them from the main object's covariate data frame
if (is.null(current_cell_names)) {
    message("⚠️ Matrix missing colnames. Fetching from sceptre_object...")
    # The covariate data frame ALWAYS has the cell IDs as rownames
    current_cell_names <- rownames(sceptre_object_highmoi@covariate_data_frame)
    
    # Safety check: lengths must match
    if (length(current_cell_names) != ncol(A_num)) {
        stop(paste("Mismatch! Matrix has", ncol(A_num), "cols but object has", length(current_cell_names), "cells."))
    }
    colnames(A_num) <- current_cell_names
}

# If guide names are missing, they should match the unique values in your guide mapping
if (is.null(current_guide_names)) {
    message("⚠️ Matrix missing rownames. Fetching from guide mapping...")
    # Get unique targets from the mapping table you loaded earlier
    # Note: This assumes the matrix rows are sorted/aligned to these unique targets.
    # In 'mixture' method, rows usually correspond to the 'grna_target' labels.
    
    # SAFE FALLBACK: If we can't be 100% sure of order, we warn the user.
    # But usually, they match the 'grna_target' column from your input.
    unique_targets <- sort(unique(guide_mapping$grna_target))
    
    if (length(unique_targets) == nrow(A_num)) {
         rownames(A_num) <- unique_targets
    } else {
         warning("Could not automatically align target names. Using generic IDs.")
         rownames(A_num) <- paste0("target_", seq_len(nrow(A_num)))
    }
}
# ---------------------------------

# 3. Convert to Long Format
s <- summary(A_num)
assign_long <- data.frame(
    cell_id = colnames(A_num)[s$j],
    grna_target = rownames(A_num)[s$i],
    value = s$x
)

# 4. Save
out_path <- file.path(data_dir, "grna_assignments_long.csv")
write.csv(assign_long, out_path, row.names = FALSE)

message("✅ Saved long CSV to: ", out_path)
print(head(assign_long))

In [ ]:
%%R
png(file.path(data_dir, "MOI.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "MOI.png"))

In [ ]:
%%R
sceptre_object_highmoi <- run_qc(
  sceptre_object = sceptre_object_highmoi,
  p_mito_threshold = 0.25
)

png(file.path(data_dir, "QC.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "QC.png"))

In [ ]:
%%R
# Calibration check


sceptre_object_highmoi <- run_calibration_check(
  sceptre_object = sceptre_object_highmoi,
  parallel = FALSE
)

png(file.path(data_dir, "calibration_check.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "calibration_check.png"))


In [ ]:
%%R
calibration_result <- get_result(
  sceptre_object = sceptre_object_highmoi,
  analysis = "run_calibration_check"
)
head(calibration_result)

In [ ]:
%%R
sceptre_object_highmoi <- run_discovery_analysis(
  sceptre_object = sceptre_object_highmoi,
  parallel = FALSE
)

png(file.path(data_dir, "discovery.png"), width = 2400, height = 2000, res = 300)

print(plot(sceptre_object_highmoi))

dev.off()

message("✅ Saved PNG plot to: ", file.path(data_dir, "discovery.png"))

In [ ]:
%%R
discovery_result <- get_result(
  sceptre_object = sceptre_object_highmoi,
  analysis = "run_discovery_analysis"
)

write.csv(discovery_result, file.path(data_dir, "TargetKD.csv"), row.names = FALSE)
message("✅ Saved discovery to: ", file.path(data_dir, "TargetKD.csv"))


### Plotting